# Step 2a: Data Cleaning - Python Baseline Preprocessing

## Overview
This notebook implements the syntactic cleaning phase for dimensional tables extracted in Step 1.

**Input Files**: 
- `data/raw/cities.csv` (270 unique cities)

**Output Files**: 
- `data/raw/cities_cleaned_step2a.csv`

**Next Steps**: 
- Cities will proceed to Step 2b (OpenRefine Wikidata reconciliation)

## Setup: Imports and Configuration

In [ ]:
import pandas as pd
import re
import string
import os

# Configure display
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Load Input Data

In [24]:
# Load dimension tables from Step 1
df_cities = pd.read_csv('../data/raw/cities.csv')

print(f"✓ Loaded cities.csv: {len(df_cities)} rows")

✓ Loaded cities.csv: 270 rows


## 1: Define City Cleaning Function

Comprehensive syntactic cleaning for historical MARC city names.

In [29]:
def clean_historical_city(city_raw):
    """
    Step 2a: Python Baseline Preprocessing (Syntactic Cleaning)
    Specialized function for cleaning KBR historical city data in MARC format.
    
    Stages:
    1. Handle NULL values
    2. Identify and standardize ambiguous/unknown entries (multilingual)
    3. Remove MARC cataloging symbols ([], (), ?, ")
    4. Strip historical publication description prefixes (A, Tot, En, et se vend, etc.)
    5. Normalize whitespace and apply Title Case
    6. Preserve internal punctuation (St., hyphens in place names)
    
    Args:
        city_raw: Raw city name string from MARC 264 field
        
    Returns:
        str: Cleaned, standardized city name
    """
    
    # Handle NaN or empty values
    if pd.isna(city_raw):
        return "Unknown"
    
    city = str(city_raw).strip()
    
    # ============================================================
    # Stage 1: Identify Ambiguous Entries (Multilingual Unknown)
    # ============================================================
    # Patterns for "publication place not identified" across multiple languages
    unknown_patterns = [
        r'non identifié', r'niet geïdentificeerd', r'not identified',
        r'non identifie', r'niet geidentificeerd', 
        r'lieu de publication', r'plaats van', r'place of',
        r'lieu non', r'plaats niet', r'place of publication'
    ]
    
    # 【關鍵修正】：使用 re.escape(pat) 確保這些字串被視為純文字比對，防止正則表達式語法噴錯
    if any(re.search(re.escape(pat), city, re.IGNORECASE) for pat in unknown_patterns):
        return "Unknown"
    
    # ============================================================
    # Stage 2: Remove Historical Publication Description Prefixes
    # ============================================================
    stopwords_prefixes = [
        r'(?i)^A\s+', r'(?i)^Tot\s+', r'(?i)^En\s+', 
        r'(?i)^et se vend(e)? a\s+', r'(?i)^et se trouve à\s+', r'(?i)^et se trouve a\s+',
        r'(?i)^"?Herdruckt,\s*'
    ]
    for pattern in stopwords_prefixes:
        city = re.sub(pattern, '', city)

    # ============================================================
    # Stage 3: Remove MARC Editorial Symbols
    # ============================================================
    # Remove brackets [], parentheses (), question marks ?, quotation marks "
    city = re.sub(r'[\[\]\(\)\?"]', '', city)

    # ====================================================
    # Stage 3b: Latin Place Name Normalization
    # ====================================================
    latin_mapping = {
        r'(?i)\bLipsiae\b': 'Lipsia',     
        r'(?i)\bRomae\b': 'Roma',         
        r'(?i)\bBruxellis\b': 'Bruxelles', 
        r'(?i)\bParisiis\b': 'Paris',     
        r'(?i)\bMechliniae\b': 'Mechlinia', 
        r'(?i)\bHannoverae\b': 'Hannover', 
    }
    for lat_pat, target in latin_mapping.items():
        city = re.sub(lat_pat, target, city)
    
    # ============================================================
    # Stage 4: Canonical Normalization
    # ============================================================
    # Remove trailing periods
    city = re.sub(r'\.$', '', city)
    
    # Collapse multiple spaces into single space
    city = re.sub(r'\s+', ' ', city).strip()
    
    # Apply Title Case
    city = string.capwords(city)
    
    # Correct missing capitalization issues with string.capwords for hyphenated and apostrophized place names
    city = re.sub(r'(-)([a-z])', lambda m: m.group(1) + m.group(2).upper(), city)
    city = re.sub(r"(')([a-z])", lambda m: m.group(1) + m.group(2).upper(), city)

    # Correct Title Case issues with French hyphenated place names
    city = city.replace('-Sur-', '-sur-')
    city = city.replace('-Le-', '-le-')
    city = city.replace('-Lez-', '-lez-')
    city = city.replace('-Et-', '-et-')
    city = city.replace('-Du-', '-du-')
    
    # Final check: if result is empty after cleaning
    if city == "" or city == "." or city.isspace():
        return "Unknown"
    
    return city

print("✓ City cleaning function redefined with bug fixes")

✓ City cleaning function redefined with bug fixes


## 2: Clean Cities

In [30]:
print("\n=== CLEANING CITIES ===")
print(f"Input: {len(df_cities)} cities")

# Apply cleaning function
df_cities['city_name_cleaned'] = df_cities['city_name'].apply(clean_historical_city)

# Check for changes
changes = (df_cities['city_name'] != df_cities['city_name_cleaned']).sum()
print(f"Changes made: {changes} entries modified")

# Replace original column
df_cities = df_cities[['city_name_cleaned']].rename(columns={'city_name_cleaned': 'city_name'})

# Remove duplicates
duplicates_before = len(df_cities)
df_cities = df_cities.drop_duplicates()
duplicates_removed = duplicates_before - len(df_cities)

print(f"Duplicates removed: {duplicates_removed}")
print(f"Output: {len(df_cities)} unique cities")
print(f"\nSample cleaned cities:")
print(df_cities.head(20))


=== CLEANING CITIES ===
Input: 270 cities
Changes made: 48 entries modified
Duplicates removed: 31
Output: 239 unique cities

Sample cleaned cities:
            city_name
0   Santiago De Chilé
1               Halle
2              Brugge
3           Amsterdam
4   Chalons-sur-Marne
5          Purmerende
6          Strasbourg
7            Courtrai
8             Avignon
9             Valence
10            Marburg
11              Tokio
12  Fontenay-le-Comte
13            Auxerre
14             Zürich
15             Keulen
16              Aalst
17             Bruges
18          Bruxelles
19               Roma


## 3: Data Quality Validation

In [31]:
print("\n=== DATA QUALITY VALIDATION ===")

# Check for NULL values
print("\nCities - NULL value check:")
print(f"  city_name NULL values: {df_cities['city_name'].isna().sum()}")

# Check for unexpected strings
print("\nUnexpected values check:")
suspicious_cities = df_cities[df_cities['city_name'].str.contains(r'[\[\]\(\)]', regex=True, na=False)]
print(f"  Cities with remaining brackets: {len(suspicious_cities)}")
if len(suspicious_cities) > 0:
    print(f"    {suspicious_cities['city_name'].tolist()}")

# Show unknown entries
unknown_cities = df_cities[df_cities['city_name'] == 'Unknown']
print(f"\nUnknown city entries: {len(unknown_cities)}")
if len(unknown_cities) > 0:
    print("  (These represent records with unidentifiable publication places)")


=== DATA QUALITY VALIDATION ===

Cities - NULL value check:
  city_name NULL values: 0

Unexpected values check:
  Cities with remaining brackets: 0

Unknown city entries: 1
  (These represent records with unidentifiable publication places)


## 4: Summary Statistics

In [32]:
print("\n=== STEP 2A SUMMARY ===")
print(f"\nCities:")
print(f"  Input rows: 270")
print(f"  Output rows: {len(df_cities)}")
print(f"  Cleaned entries: {270 - len(df_cities)}")

print(f"\n✓ Step 2a_Cities preprocessing complete")


=== STEP 2A SUMMARY ===

Cities:
  Input rows: 270
  Output rows: 239
  Cleaned entries: 31

✓ Step 2a_Cities preprocessing complete


## 5: Export Cleaned Data

In [33]:
# Define output directory
output_dir = '../data/raw/'
os.makedirs(output_dir, exist_ok=True)

# Export cleaned CSV files
df_cities.to_csv(f'{output_dir}cities_cleaned_step2a.csv', index=False, encoding='utf-8')

print("\n=== FILES EXPORTED ===")
print(f"✓ cities_cleaned_step2a.csv ({len(df_cities)} rows)")
print(f"\nExport location: {output_dir}")
print(f"\nNext: Proceed to Step 2b (OpenRefine Wikidata reconciliation)")


=== FILES EXPORTED ===
✓ cities_cleaned_step2a.csv (239 rows)

Export location: ../data/raw/

Next: Proceed to Step 2b (OpenRefine Wikidata reconciliation)
